In [2]:
import os
from pyspark.sql import SparkSession

# Build the local cluster connection context matching your localhost requirements
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Prakash_Spark_Assignment") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

# Reference your exact file path seen in image_54d207.png
data_path = os.path.join("..", "data", "sample_transactions.csv")

# Read the local data cleanly
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(data_path)

print("--- RAW DATA INGESTION ---")
df_raw.show()

26/06/22 09:43:05 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


--- RAW DATA INGESTION ---
+---+-------+----+--------------+--------------+------+
| id|   name| age|      category|        region|amount|
+---+-------+----+--------------+--------------+------+
|  1|  Alice|  28|   Electronics|         North|1200.0|
|  2|    Bob|  35|      Clothing|         South| 450.0|
|  3|Charlie|NULL|   Electronics|          East| 700.0|
|  4|  David|  42|   CLOTHING   |          West| 300.0|
|  1|  Alice|  28|   Electronics|         North|1200.0|
|  5|    Eva|  19|          Home|         North|  NULL|
|  6|  Frank|  50|   Electronics|         South|2200.0|
|  7|  Grace|  65|          Home|invalid_region| 150.0|
+---+-------+----+--------------+--------------+------+



In [3]:
# A. Remove perfect duplicate rows (drops the repeating Alice row)
df_deduped = df_raw.dropDuplicates()

# B. Handle missing fields safely
# 1. Fill Charlie's missing age row with a default of 30
# 2. Drop Eva's row entirely since her amount field is completely empty/NULL
df_cleaned = df_deduped.na.fill({"age": 30}).na.drop(subset=["amount"])

print("--- DATA AFTER DUPLICATE & NULL CLEANING ---")
df_cleaned.show()

--- DATA AFTER DUPLICATE & NULL CLEANING ---
+---+-------+---+--------------+--------------+------+
| id|   name|age|      category|        region|amount|
+---+-------+---+--------------+--------------+------+
|  6|  Frank| 50|   Electronics|         South|2200.0|
|  7|  Grace| 65|          Home|invalid_region| 150.0|
|  1|  Alice| 28|   Electronics|         North|1200.0|
|  4|  David| 42|   CLOTHING   |          West| 300.0|
|  2|    Bob| 35|      Clothing|         South| 450.0|
|  3|Charlie| 30|   Electronics|          East| 700.0|
+---+-------+---+--------------+--------------+------+



In [4]:
from pyspark.sql.functions import col, lower, trim
from pyspark.sql.types import IntegerType, DoubleType

# A. Standardize messy text data by trimming spaces and converting to lowercase
# B. Cast your metrics columns into strict mathematical data types
# C. Rename the amount column to 'sales_amount' to standardize your metrics
df_schema_mod = df_cleaned \
    .withColumn("category", lower(trim(col("category")))) \
    .withColumn("region", lower(trim(col("region")))) \
    .withColumn("age", col("age").cast(IntegerType())) \
    .withColumn("amount", col("amount").cast(DoubleType())) \
    .withColumnRenamed("amount", "sales_amount")

print("--- SCHEMA TRANSFORMS APPLIED ---")
df_schema_mod.show()
df_schema_mod.printSchema()

--- SCHEMA TRANSFORMS APPLIED ---


[Stage 11:>                                                         (0 + 1) / 1]

+---+-------+---+-----------+--------------+------------+
| id|   name|age|   category|        region|sales_amount|
+---+-------+---+-----------+--------------+------------+
|  6|  Frank| 50|electronics|         south|      2200.0|
|  7|  Grace| 65|       home|invalid_region|       150.0|
|  1|  Alice| 28|electronics|         north|      1200.0|
|  4|  David| 42|   clothing|          west|       300.0|
|  2|    Bob| 35|   clothing|         south|       450.0|
|  3|Charlie| 30|electronics|          east|       700.0|
+---+-------+---+-----------+--------------+------------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = false)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_amount: double (nullable = true)



In [5]:
# Isolate rows using your targeted logical rules (e.g., target active working ages, prune out the invalid_region)
df_filtered = df_schema_mod.filter(
    (col("age") >= 20) & 
    (col("age") <= 60) & 
    (col("region") != "invalid_region")
)

print("--- DATA FILTERING ACTIVE ---")
df_filtered.show()

--- DATA FILTERING ACTIVE ---
+---+-------+---+-----------+------+------------+
| id|   name|age|   category|region|sales_amount|
+---+-------+---+-----------+------+------------+
|  6|  Frank| 50|electronics| south|      2200.0|
|  1|  Alice| 28|electronics| north|      1200.0|
|  4|  David| 42|   clothing|  west|       300.0|
|  2|    Bob| 35|   clothing| south|       450.0|
|  3|Charlie| 30|electronics|  east|       700.0|
+---+-------+---+-----------+------+------------+

